# Proyecto Integrador – Fase 1: Predicción del precio de viviendas

**Curso:** Modelos y Simulación de Sistemas I  
**Problema:** Regresión supervisada  
**Variable objetivo:** `price`

El objetivo es construir un modelo capaz de predecir el precio de venta de una vivienda a partir de sus características físicas y de ubicación. El proceso incluye análisis exploratorio, preparación de datos, prevención de fuga de información, construcción de un modelo base, comparación de modelos y almacenamiento del modelo final.

**Fuente del dataset:** Kaggle – *House Price Prediction*.

## 1. Importación de librerías y carga del conjunto de datos

Para garantizar la reproducibilidad, se fija `random_state=42` en los procedimientos aleatorios. El archivo `DataSet_House_Price_Prediction.csv` debe estar ubicado junto al notebook dentro de la carpeta `fase-1`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Buscar el CSV en ubicaciones relativas habituales.
rutas_posibles = [
    Path("DataSet_House_Price_Prediction.csv"),
    Path("fase-1") / "DataSet_House_Price_Prediction.csv"
]

ruta_csv = next((ruta for ruta in rutas_posibles if ruta.exists()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró DataSet_House_Price_Prediction.csv. "
        "Ubique el CSV junto al notebook dentro de la carpeta fase-1."
    )

df_original = pd.read_csv(ruta_csv)
df = df_original.copy()

print("Archivo cargado:", ruta_csv)
print("Dimensiones originales:", df.shape)
display(df.head())

## 2. Descripción y análisis exploratorio

El conjunto de datos contiene información sobre viviendas, incluyendo dimensiones, número de habitaciones y baños, año de construcción, condición y ubicación. Se revisan dimensiones, tipos de variables, estadísticas descriptivas, valores faltantes y distribución de variables relevantes.

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
display(df.dtypes.to_frame("tipo"))

print("\nValores faltantes originales por variable:")
display(df.isna().sum().to_frame("nulos"))

print("\nEstadísticas descriptivas:")
display(df.describe(include="all").T)

In [ ]:
# Distribuciones de variables numéricas relevantes
variables_graficas = ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"]

for variable in variables_graficas:
    plt.figure(figsize=(7, 4))
    plt.hist(df[variable].dropna(), bins=30)
    plt.title(f"Distribución de {variable}")
    plt.xlabel(variable)
    plt.ylabel("Frecuencia")
    plt.show()

    plt.figure(figsize=(7, 2.5))
    plt.boxplot(df[variable].dropna(), vert=False)
    plt.title(f"Boxplot de {variable}")
    plt.xlabel(variable)
    plt.show()

In [ ]:
# Correlación entre variables numéricas
numericas_eda = df.select_dtypes(include=np.number)
correlaciones = numericas_eda.corr()

print("Correlación de las variables numéricas con price:")
display(
    correlaciones["price"]
    .drop("price")
    .sort_values(ascending=False)
    .to_frame("correlacion_con_price")
)

### Observaciones del EDA

Los boxplots muestran valores extremos en varias variables. No se eliminan automáticamente porque una vivienda de gran tamaño o alto precio puede ser una observación real. En cambio, se revisa específicamente la variable objetivo y se detectan registros con `price = 0`, que no representan un precio de venta útil para entrenar el modelo.

También se evita utilizar `price_per_sqft` como predictor porque se calcula a partir de `price`; incluirla introduciría información de la variable objetivo y produciría fuga de información.

In [ ]:
print("Registros totales:", len(df))
print("Registros con price = 0:", (df["price"] == 0).sum())

print("\nCasos con mayor price_per_sqft:")
display(
    df[["price", "sqft_living", "price_per_sqft"]]
    .sort_values("price_per_sqft", ascending=False)
    .head(10)
)

## 3. Valores faltantes y limpieza

El dataset original no presenta valores faltantes. Para cumplir el requisito de analizar y tratar datos faltantes, se introduce de forma reproducible aproximadamente un 1% de valores faltantes en `sqft_living`.

La imputación **no se realiza todavía**. Primero se separarán entrenamiento y prueba; posteriormente la mediana será aprendida exclusivamente a partir del conjunto de entrenamiento mediante un `Pipeline`, evitando fuga de información.

In [ ]:
df = df_original.copy()

# Introducir aproximadamente 1% de valores faltantes en sqft_living
np.random.seed(RANDOM_STATE)
cantidad_nulos = int(len(df) * 0.01)
filas_nulas = np.random.choice(df.index, cantidad_nulos, replace=False)
df.loc[filas_nulas, "sqft_living"] = np.nan

print("Valores faltantes introducidos en sqft_living:", df["sqft_living"].isna().sum())
print("Porcentaje:", round(df["sqft_living"].isna().mean() * 100, 2), "%")

# Eliminar registros con precio igual a cero
df = df[df["price"] > 0].copy()

print("\nDimensiones después de retirar price = 0:", df.shape)
print("Valores faltantes restantes en sqft_living:", df["sqft_living"].isna().sum())

## 4. Preparación de variables

`price` es la variable objetivo. `price_per_sqft` se elimina para evitar fuga de información. La fecha se transforma en año y mes de venta. `street` se elimina por su cardinalidad extremadamente alta, mientras que `city` y `statezip` se conservan y se codifican mediante One-Hot Encoding.

In [ ]:
y = df["price"]
X = df.drop(columns=["price", "price_per_sqft"]).copy()

# Transformar la fecha
X["date"] = pd.to_datetime(X["date"])
X["year_sale"] = X["date"].dt.year
X["month_sale"] = X["date"].dt.month
X = X.drop(columns=["date"])

print("Cardinalidad de variables de ubicación:")
for columna in ["street", "city", "statezip"]:
    print(f"{columna}: {X[columna].nunique()} categorías")

# street presenta casi una categoría por vivienda
X = X.drop(columns=["street"])

print("\nNúmero final de variables predictoras:", X.shape[1])
print(X.columns.tolist())

## 5. Separación entrenamiento/prueba

Se utiliza una división aleatoria 80/20 con semilla 42. El conjunto de prueba se mantiene separado durante el entrenamiento y las transformaciones que aprenden de los datos se ajustan únicamente con entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("Nulos en entrenamiento:", X_train.isna().sum().sum())
print("Nulos en prueba:", X_test.isna().sum().sum())

## 6. Preprocesamiento y prevención de fuga de información

Las variables categóricas se codifican con `OneHotEncoder(handle_unknown="ignore")`. Las variables numéricas pasan por `SimpleImputer(strategy="median")`.

Al incluir la imputación y la codificación dentro de `Pipeline`, ambos transformadores se ajustan durante `fit()` utilizando únicamente `X_train`. De esta forma, el conjunto de prueba no aporta información al entrenamiento.

In [ ]:
categoricas = ["city", "statezip"]
numericas = [col for col in X_train.columns if col not in categoricas]

pipeline_numerico = Pipeline(steps=[
    ("imputacion", SimpleImputer(strategy="median"))
])

preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("num", pipeline_numerico, numericas)
    ]
)

print("Variables categóricas:", len(categoricas))
print("Variables numéricas:", len(numericas))

## 7. Modelo base (baseline)

Antes del modelo predictivo se construye un `DummyRegressor` que siempre predice la mediana del precio observada en entrenamiento. La mediana se utiliza como referencia simple y robusta frente a precios extremos.

In [ ]:
modelo_base = DummyRegressor(strategy="median")
modelo_base.fit(X_train[["bedrooms"]], y_train)
y_pred_base = modelo_base.predict(X_test[["bedrooms"]])

mae_base = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base = r2_score(y_test, y_pred_base)

print("MODELO BASE - DummyRegressor (mediana)")
print("MAE:", round(mae_base, 2))
print("RMSE:", round(rmse_base, 2))
print("R²:", round(r2_base, 4))

## 8. Modelos predictivos

Se prueban tres alternativas: Regresión Lineal, Random Forest y Gradient Boosting. La métrica principal es **MAE**, porque expresa el error medio en las mismas unidades monetarias de la variable objetivo y es fácil de interpretar. También se reportan RMSE y R² como métricas complementarias.

In [ ]:
def evaluar_modelo(nombre, modelo):
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    return {
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    }, modelo

modelo_lineal = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", LinearRegression())
])

modelo_rf = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

modelo_gb = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("modelo", GradientBoostingRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE
    ))
])

resultados_modelos = []

res_lineal, modelo_lineal = evaluar_modelo("Regresión Lineal", modelo_lineal)
resultados_modelos.append(res_lineal)

res_rf, modelo_rf = evaluar_modelo("Random Forest", modelo_rf)
resultados_modelos.append(res_rf)

res_gb, modelo_gb = evaluar_modelo("Gradient Boosting", modelo_gb)
resultados_modelos.append(res_gb)

resultados = pd.DataFrame([
    {
        "Modelo": "Baseline - mediana",
        "MAE": mae_base,
        "RMSE": rmse_base,
        "R2": r2_base
    },
    *resultados_modelos
])

display(resultados.round(2))

## 9. Selección y evaluación del modelo final

Se selecciona **Gradient Boosting** tomando el MAE como criterio principal. La comparación debe interpretarse conjuntamente con RMSE y R²: un modelo puede obtener menor MAE y, al mismo tiempo, comportarse peor ante errores grandes.

Lo importante es verificar que el modelo predictivo supere claramente al baseline y documentar las limitaciones observadas.

In [ ]:
# El modelo Gradient Boosting ya fue entrenado en la sección anterior.
modelo_final = modelo_gb
y_pred_final = modelo_final.predict(X_test)

mae_final = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2_final = r2_score(y_test, y_pred_final)

print("MODELO FINAL - Gradient Boosting")
print("MAE:", round(mae_final, 2))
print("RMSE:", round(rmse_final, 2))
print("R²:", round(r2_final, 4))

print("\nComparación con baseline:")
print("Mejora de MAE:", round(mae_base - mae_final, 2))

### Interpretación

El MAE indica el error absoluto promedio de las predicciones en las mismas unidades de `price`. El RMSE penaliza con mayor fuerza los errores grandes, mientras que R² indica qué proporción de la variabilidad de los precios es explicada por el modelo en el conjunto de prueba.

Las principales dificultades del problema son la presencia de precios extremos, variables categóricas de ubicación con muchas categorías y la necesidad de evitar variables derivadas del precio que producirían fuga de información. En futuras versiones podrían evaluarse ajustes de hiperparámetros, nuevas variables de ubicación y estrategias adicionales para manejar valores extremos.

## 10. Almacenamiento y reutilización del modelo

El pipeline completo se guarda con `joblib`. Posteriormente se carga nuevamente y se utiliza para generar una predicción, demostrando que el modelo almacenado puede reutilizarse sin volver a entrenarlo.

In [ ]:
joblib.dump(modelo_final, "modelo.joblib")

modelo_cargado = joblib.load("modelo.joblib")
ejemplo = X_test.iloc[[0]]
prediccion = modelo_cargado.predict(ejemplo)

print("Modelo guardado y cargado correctamente.")
print("Precio real:", round(float(y_test.iloc[0]), 2))
print("Precio predicho:", round(float(prediccion[0]), 2))

## 11. Conclusiones

Se construyó un flujo reproducible para un problema de regresión de precios de viviendas. El proceso incluyó análisis exploratorio, tratamiento de valores faltantes sin utilizar información del conjunto de prueba, prevención de fuga de información, construcción de un baseline, comparación de modelos y almacenamiento del modelo final.

La prioridad de esta fase es mantener un procedimiento técnicamente consistente, comprensible y reutilizable para las siguientes etapas del proyecto.